## Otimização de Hiperparâmetros - XGBOOST 
#### preprocessador v3

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
import time

#xgboost
from xgboost import XGBClassifier

# sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, KFold, cross_validate, cross_val_score, RandomizedSearchCV, StratifiedKFold


from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score,
                             average_precision_score,roc_curve)
from sklearn.base import BaseEstimator, TransformerMixin, clone

from scipy.stats import ttest_rel
from scipy.stats import randint, uniform

#hiperparamentros search
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from scipy.stats import randint, uniform,loguniform


# Importações locais
from setup_notebook import setup_path
setup_path()
from src.model_utils import *
from src.preprocess_utils_diab3 import * #(NOVO atualizações)

print("#Processo iniciado em:", time.strftime("%H:%M:%S"))
start_inicial = time.time()

#Processo iniciado em: 11:07:43


## 1. Load Data & Pipeline

In [2]:
BASE = Path.cwd().parent

PP3 = joblib.load(BASE/'src'/'preprocess_diabetes_v3.joblib')['preprocessador']

DATA_DIR = BASE/"data"/"raw"
X_train = pd.read_csv(DATA_DIR/"X_train_raw.csv")
X_test  = pd.read_csv(DATA_DIR/"X_test_raw.csv")
y_train = pd.read_csv(DATA_DIR/"y_train_raw.csv").values.ravel()
y_test  = pd.read_csv(DATA_DIR/"y_test_raw.csv").values.ravel()
mtd_scoring='roc_auc'

## 2.Baseline

In [3]:
print("#Processo iniciado em:", time.strftime("%H:%M:%S"))

# BASELINE
model_base = XGBClassifier(objective='binary:logistic',eval_metric='logloss',enable_categorical=True,
                           tree_method="hist" , random_state=42)
pipe_base  = pipe_models(model_base, PP3)
pipe_base.fit(X_train, y_train)

# =====================================================
# 1) Predição
# =====================================================
y_pred_base=pipe_base.predict(X_test)

# =====================================================
# 2) Otimização do threshold de decisão
# =====================================================
# Como classificadores probabilísticos usam threshold padrão de 0.5,
# testamos diferentes valores para verificar se existe um ponto que
# maximize a acurácia no conjunto de teste.
best_th_base,max_acc_base,y_probs_base=best_threshold2(pipe_base, X_train, y_train,X_test,y_test)

# =====================================================
# 3) Desempenho em validação cruzada
# =====================================================
# A validação cruzada utiliza a mesma métrica definida
# no processo de tuning para avaliar a estabilidade do modelo.
cv_s = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores_base = cross_val_score(pipe_base, X_train, y_train,
                                cv=cv_s,
                                scoring=mtd_scoring,
                                n_jobs=-1)

print(f"\n{'='*70}")
print(f" 📍 RESULTADOS BASELINE".center(70))
print(f"{'='*70}")

# =====================================================
# 4) Avaliação por validação cruzada (Treino)
# =====================================================

print("📊 CROSS-VALIDATION")
print(f"   Média {mtd_scoring}:       {cv_scores_base.mean():>15.4f} ± {cv_scores_base.std():.4f}")

# =====================================================
# 5) Avaliação no conjunto de teste
# =====================================================
print(f"\n✅ TEST SET")
print(f"   Padrão (0.5):              {accuracy_score(y_test, y_pred_base):>10.4f}")
print(f"   Otimizado:                 {max_acc_base:>10.4f} (threshold ={best_th_base:>6.3f})")
print(f"   ROC-AUC:                   {roc_auc_score(y_test, y_probs_base):>10.4f}")
print(f"   Avg precision:             {average_precision_score(y_test, y_probs_base):>10.4f}")

# =====================================================
# 6) Hiperparametros utilizados
# =====================================================
print(f"{'='*70}")
print_hiper(model_base.get_params())
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))

#Processo iniciado em: 11:07:44

                         📍 RESULTADOS BASELINE                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.7207 ± 0.0015

✅ TEST SET
   Padrão (0.5):                  0.6818
   Otimizado:                     0.6815 (threshold = 0.510)
   ROC-AUC:                       0.7228
   Avg precision:                 0.8084
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Objective                 : binary:logistic
• Base Score                : None
• Booster                   : None
• Callbacks                 : None
• Colsample Bylevel         : None
• Colsample Bynode          : None
• Colsample Bytree          : None
• Device                    : None
• Early Stopping Rounds     : None
• Enable Categorical        : True
• Eval Metric               : logloss
• Feature Types             : None
• Gamma                     : None
• Grow Policy               : None
• Importance Type           : None
• Interaction C

## 3.Buscas por hiperparamentros


### 3.a Exploratório


In [4]:
print("#Processo Iniciado em:", time.strftime("%H:%M:%S"))
# Exploratório
param_dist_1 = {
    # 1. Controle de Aprendizado
    'model__n_estimators': randint(100, 3000), 
    'model__learning_rate':  loguniform(0.003, 0.3),
    
    # 2. Complexidade e Profundidade
    'model__max_depth': randint(3, 13),
    'model__min_child_weight': randint(1, 11),
    
    # 3. Robustez contra Overfitting (Stochastic Gradient Boosting)
    'model__subsample': uniform(0.5, 0.5),
    'model__colsample_bytree': uniform(0.5, 0.5),
    
    # 4. Regularização (Penalidades)
    'model__gamma': uniform(0, 0.5),
    'model__reg_alpha': uniform(0, 5), 
    'model__reg_lambda': uniform(1, 8)   
}

cv_s = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
numero_itera=50


search_1 = RandomizedSearchCV(
    pipe_base, param_dist_1,
    n_iter=numero_itera, cv=cv_s,
    scoring=mtd_scoring,
    random_state=42, verbose=1
)

start = time.time()
search_1.fit(X_train, y_train)
end = time.time()

best_1 = search_1.best_estimator_


# -----------------------------------------------------
# 1) Predição
# -----------------------------------------------------
# DATA_MODELS= BASE /"models"
y_pred_1=best_1.predict(X_test)

# -----------------------------------------------------
# 2) Otimização do threshold de decisão
# -----------------------------------------------------
best_th_1,max_acc_1,y_probs_1=best_threshold2(best_1, X_train, y_train,X_test,y_test)

print("\n#Finalizando a validação cruzada em:", time.strftime("%H:%M:%S"))

# -----------------------------------------------------
# 3) Desempenho em validação cruzada
# -----------------------------------------------------
cv_scores_1 = cross_val_score(best_1, X_train, y_train,
                                cv=cv_s,
                                scoring=mtd_scoring,
                                n_jobs=-1) #90s

print(f"\n{'='*70}")
print(f" 📍 RESULTADOS SEARCH 1".center(70))
print(f"{'='*70}")

# -----------------------------------------------------
# 4) Avaliação por validação cruzada (Treino)
# -----------------------------------------------------
print("📊 CROSS-VALIDATION")
print(f"   Média {mtd_scoring}:       {cv_scores_1.mean():>15.4f} ± {cv_scores_1.std():.4f}")

# -----------------------------------------------------
# 5) Avaliação no conjunto de teste
# -----------------------------------------------------
print(f"\n✅ TEST SET")
print(f"   Padrão (0.5):              {accuracy_score(y_test, y_pred_1):>10.4f}")
print(f"   Otimizado:                 {max_acc_1:>10.4f} (threshold ={best_th_1:>6.3f})")
print(f"   ROC-AUC:                   {roc_auc_score(y_test, y_probs_1):>10.4f}")
print(f"   Avg precision:             {average_precision_score(y_test, y_probs_1):>10.4f}")

#-----------------------------------------------------
# 6) Hiperparametros utilizados
#-----------------------------------------------------
print(f"{'='*70}")
print_hiper(search_1.best_params_)
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))
joblib.dump(search_1, 'temp_1.joblib')


#Processo Iniciado em: 11:07:57
Fitting 3 folds for each of 50 candidates, totalling 150 fits

#Finalizando a validação cruzada em: 12:31:11

                         📍 RESULTADOS SEARCH 1                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.7271 ± 0.0013

✅ TEST SET
   Padrão (0.5):                  0.6856
   Otimizado:                     0.6853 (threshold = 0.510)
   ROC-AUC:                       0.7272
   Avg precision:                 0.8118
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Colsample Bytree          : 0.9077307142274171
• Gamma                     : 0.35342867192380856
• Learning Rate             : 0.08612626044586402
• Max Depth                 : 3
• Min Child Weight          : 5
• N Estimators              : 2378
• Reg Alpha                 : 3.2553851275097223
• Reg Lambda                : 8.319677404350246
• Subsample                 : 0.9250192888948996
--------------------------------------------------


['temp_1.joblib']

### 3.b Refine


In [5]:
print("#Processo Iniciado em:", time.strftime("%H:%M:%S"))
def generate_refined_grid(best_params):
    """
    Gera um grid refinado automaticamente baseado nos melhores parâmetros
    do Random Search exploratório (XGBoost).
    """
    # ---------------------------
    # 1. n_estimators
    # ---------------------------
    n_best = best_params['model__n_estimators']
    n_min  = int(n_best * 0.8)
    n_max  = int(n_best * 1.2)

    # ---------------------------
    # 2. learning_rate
    # ---------------------------
    lr_best = best_params['model__learning_rate']
    lr_st   = lr_best * 0.75
    lr_dist = 0.3  # Amplitude definida no seu código original

    # ---------------------------
    # 3. max_depth (Tratamento de None)
    # ---------------------------
    depth_best = best_params['model__max_depth']
    if depth_best is not None:
        depth_list = [max(1, depth_best - 1), depth_best, depth_best + 1]
    else:
        depth_list = [None, 10, 20]

    # ---------------------------
    # 4. min_child_weight
    # ---------------------------
    mcw_best = best_params['model__min_child_weight']
    mcw_list = [max(0, mcw_best - 1), mcw_best, mcw_best + 1]

    # ---------------------------
    # 5. subsample
    # ---------------------------
    ss_best = best_params['model__subsample']
    ss_st   = ss_best * 0.75
    ss_dist = 0.3

    # ---------------------------
    # 6. colsample_bytree
    # ---------------------------
    cb_best = best_params['model__colsample_bytree']
    cb_st   = cb_best * 0.75
    cb_dist = 0.3

    # ---------------------------
    # 7. gamma
    # ---------------------------
    gamma_best = best_params['model__gamma']
    gamma_st   = gamma_best * 0.75
    gamma_dist = 0.5

    # ---------------------------
    # 8. regularização (alpha e lambda)
    # ---------------------------
    alpha_best  = best_params['model__reg_alpha']
    lambda_best = best_params['model__reg_lambda']

    alpha_st   = alpha_best * 0.75
    alpha_dist = 0.5

    lambda_st   = lambda_best * 0.75
    lambda_dist = 0.5

    # ---------------------------
    # PRINTS DIAGNÓSTICOS
    # ---------------------------
    print(f"{'='*30} NOVO GRID DE REFINAMENTO (XGBoost) {'='*30}")
    print(f"🔹 n_estimators        : {n_min} - {n_max}")
    print(f"🔹 learning_rate       : {lr_st:.5f} - {lr_st + lr_dist:.5f}")
    print(f"🔹 max_depth           : {depth_list}")
    print(f"🔹 min_child_weight    : {mcw_list}")
    print(f"🔹 subsample           : {ss_st:.3f} - {min(1.0, ss_st + ss_dist):.3f}")
    print(f"🔹 colsample_bytree    : {cb_st:.3f} - {min(1.0, cb_st + cb_dist):.3f}")
    print(f"🔹 gamma               : {gamma_st:.4f} - {gamma_st + gamma_dist:.4f}")
    print(f"🔹 reg_alpha           : {alpha_st:.4f} - {alpha_st + alpha_dist:.4f}")
    print(f"🔹 reg_lambda          : {lambda_st:.4f} - {lambda_st + lambda_dist:.4f}")
    print(f"{'='*86}\n")

    # ---------------------------
    # GRID FINAL
    # ---------------------------
    refined_grid = {
        'model__n_estimators': randint(n_min, n_max + 1),
        'model__learning_rate': uniform(lr_st, lr_dist),
        'model__max_depth': depth_list,
        'model__min_child_weight': mcw_list,
        'model__subsample': uniform(ss_st, ss_dist),
        'model__colsample_bytree': uniform(cb_st, cb_dist),
        'model__gamma': uniform(gamma_st, gamma_dist),
        'model__reg_alpha': uniform(alpha_st, alpha_dist),
        'model__reg_lambda': uniform(lambda_st, lambda_dist)
    }
    return refined_grid

# Aplicação automática
param_dist_2 = generate_refined_grid(search_1.best_params_)

numero_itera = 30
search_2 = RandomizedSearchCV(
    pipe_base, param_dist_2,
    n_iter=numero_itera, cv=cv_s, 
    scoring=mtd_scoring,
    random_state=42, verbose=1
)

start = time.time()
search_2.fit(X_train, y_train)
end = time.time()

best_2 = search_2.best_estimator_

# -----------------------------------------------------
# 1) Predição
# -----------------------------------------------------
y_pred_2=best_2.predict(X_test)

# -----------------------------------------------------
# 2) Otimização do threshold de decisão
# -----------------------------------------------------
best_th_2,max_acc_2,y_probs_2=best_threshold2(best_2, X_train, y_train,X_test,y_test)

print("\n#Finalizando a validação cruzada em:", time.strftime("%H:%M:%S"))

# -----------------------------------------------------
# 3) Desempenho em validação cruzada
# -----------------------------------------------------
cv_scores_2 = cross_val_score(best_2, X_train, y_train,
                                cv=cv_s,
                                scoring=mtd_scoring,
                                n_jobs=-1) #90s

print(f"\n{'='*70}")
print(f" 📍 RESULTADOS SEARCH 2".center(70))
print(f"{'='*70}")

# -----------------------------------------------------
# 4) Avaliação por validação cruzada (Treino)
# -----------------------------------------------------
print("📊 CROSS-VALIDATION")
print(f"   Média {mtd_scoring}:       {cv_scores_2.mean():>15.4f} ± {cv_scores_2.std():.4f}")

# -----------------------------------------------------
# 5) Avaliação no conjunto de teste
# -----------------------------------------------------
print(f"\n✅ TEST SET")
print(f"   Padrão (0.5):              {accuracy_score(y_test, y_pred_2):>10.4f}")
print(f"   Otimizado:                 {max_acc_2:>10.4f} (threshold ={best_th_2:>6.3f})")
print(f"   ROC-AUC:                   {roc_auc_score(y_test, y_probs_2):>10.4f}")
print(f"   Avg precision:             {average_precision_score(y_test, y_probs_2):>10.4f}")

#-----------------------------------------------------
# 6) Hiperparametros utilizados
#-----------------------------------------------------
print(f"{'='*70}")
print_hiper(search_2.best_params_)
#print_hiper(best_2.named_steps.model.get_params())
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))
joblib.dump(search_2, 'temp_2.joblib')

#Processo Iniciado em: 12:32:28
============================== NOVO GRID DE REFINAMENTO (XGBoost) ==============================
🔹 n_estimators        : 1902 - 2853
🔹 learning_rate       : 0.06459 - 0.36459
🔹 max_depth           : [2, 3, 4]
🔹 min_child_weight    : [4, 5, 6]
🔹 subsample           : 0.694 - 0.994
🔹 colsample_bytree    : 0.681 - 0.981
🔹 gamma               : 0.2651 - 0.7651
🔹 reg_alpha           : 2.4415 - 2.9415
🔹 reg_lambda          : 6.2398 - 6.7398

Fitting 3 folds for each of 30 candidates, totalling 90 fits

#Finalizando a validação cruzada em: 13:07:38

                         📍 RESULTADOS SEARCH 2                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.7270 ± 0.0013

✅ TEST SET
   Padrão (0.5):                  0.6849
   Otimizado:                     0.6849 (threshold = 0.500)
   ROC-AUC:                       0.7272
   Avg precision:                 0.8118
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Cols

['temp_2.joblib']

In [6]:
end_final = time.time()
print(f"Tempo final: {(end_final-start_inicial)/60:.2f} min")

Tempo final: 121.14 min


## 4.Salvando modelos

In [7]:
#Salvar Hiperparametros joblib
joblib.dump(search_1.best_estimator_, 'modelo_XGB2_final_randsearch.'+mtd_scoring+'_v3.joblib')
joblib.dump(search_2.best_estimator_, 'modelo_XGB2_final_refine.'+mtd_scoring+'_v3.joblib')
print(f"✅ Arquivo  salvo • {time.strftime('%d/%m/%Y-%H:%M:%S')}")

✅ Arquivo  salvo • 07/03/2026-13:08:51
